In [1]:
import pandas as pd

In [59]:
# Load tweet sentiment data
tweet_path = r"H:\FOM- Study Material\Semester Three\Big-Data-Analysis Project\fam_momentum_index\Data\tweet_sentiment_by_minute.csv"
df_tweets = pd.read_csv(tweet_path, sep=";")

# Load audio data
audio_path = r"H:\FOM- Study Material\Semester Three\Big-Data-Analysis Project\fam_momentum_index\Data\match_audio_volume_by_second.csv"
df_audio = pd.read_csv(audio_path, sep=";")

In [61]:
# Match started at 2025-07-07 20:00:00
match_start = pd.to_datetime("2025-07-07 20:00:00")

In [63]:
print(df_audio.columns.tolist())
df_audio.columns = df_audio.columns.str.replace('\ufeff', '')

['second', 'rms_db', 'normalized_volume']


In [65]:
# Convert second to minute and align with real time
df_audio["minute"] = (df_audio["second"] // 60).astype(int)
df_audio["timestamp"] = match_start + pd.to_timedelta(df_audio["minute"], unit='m')

In [67]:
# Aggregate audio by minute
df_audio_minute = df_audio.groupby("timestamp").agg(
    avg_audio_volume=("normalized_volume", "mean")
).reset_index()

In [71]:
df_tweets["minute"] = pd.to_datetime(df_tweets["minute"])


In [73]:
# Merge audio + tweets on timestamp
df_fmi = pd.merge(df_tweets, df_audio_minute, how="left", left_on="minute", right_on="timestamp")

In [75]:
# Compute Fan Momentum Index (weighted average)
df_fmi["fmi"] = (
    0.4 * df_fmi["norm_sentiment"] +
    0.3 * df_fmi["norm_volume"] +
    0.3 * df_fmi["avg_audio_volume"]
)

# Save result
output_path = r"H:\FOM- Study Material\Semester Three\Big-Data-Analysis Project\fam_momentum_index\Data\final_fmi_dataset.csv"
df_fmi.to_csv(output_path, index=False)
print("Final FMI dataset saved.")

Final FMI dataset saved.
